# Crypto Exchange API Tester

Manual notebook for testing the project Bybit and OKX API integration paths.

Run this notebook from the `backend` directory with the backend virtual environment kernel. It uses the existing project clients and normalizers:

- `core.crypto_exchange_clients.BybitClient`
- `core.crypto_exchange_clients.OKXClient`
- `core.crypto_exchange_import.normalize_bybit_spot_execution`
- `core.crypto_exchange_import.normalize_okx_spot_fill`
- optional `core.broker_api_utils.BybitAPI` / `OKXAPI` using credentials stored in the app database

The notebook does not print API secrets and does not write transactions to the database unless `ALLOW_DB_WRITES = True` is set explicitly.

## Credential Setup

Preferred path: use credentials stored through the app UI.

1. Create a `Broker` for Bybit or OKX.
2. Create a broker account under that broker.
3. Save the exchange API credentials through `User Settings -> Broker API credentials`.
4. Run the `Database Credential Discovery` cells below.

This notebook defaults to `user_id=1`, because the local test database has the ByBit and OKX broker/account/token rows under that user. You can override that with environment variables if needed:

- `PM_USER_ID=1`
- optional `PM_ACCOUNT_ID=<account id>`
- optional `CRYPTO_API_PROVIDER=auto|bybit|okx`
- optional `CRYPTO_API_LOOKBACK_DAYS=7`

Direct environment credentials are still supported as a fallback for the direct client smoke tests, but they are not required when using stored app credentials.

Bybit key guidance:

- Create either a mainnet key or a testnet key. Testnet keys are separate from mainnet keys.
- Required values: API key and API secret.
- Required read permissions: account/wallet read, transaction log read, order/execution history read for the categories you want to test.
- Disable withdrawals and trading permissions.
- Restrict by IP if your network setup allows it.
- Environment fallback variables:
  - `BYBIT_API_KEY`
  - `BYBIT_API_SECRET`
  - `BYBIT_TESTNET=1` for testnet, otherwise `0`
  - optional `BYBIT_ACCOUNT_TYPE=UNIFIED`
  - optional `BYBIT_CATEGORY=spot`

OKX key guidance:

- Create a read-only API key. OKX also requires the passphrase created with the key.
- Required values: API key, API secret, passphrase.
- Required read permissions: account read and trade/fills history read. For rewards/transfers testing, also allow read access to funding/account bills.
- Disable withdrawals and trading permissions.
- Restrict by IP if your network setup allows it.
- For demo/simulated testing, create or use demo trading credentials and set `OKX_SIMULATED_TRADING=1`.
- Environment fallback variables:
  - `OKX_API_KEY`
  - `OKX_API_SECRET`
  - `OKX_PASSPHRASE`
  - `OKX_SIMULATED_TRADING=1` for demo/simulated trading, otherwise `0`


In [ ]:
import os
import sys
from datetime import datetime, timedelta, timezone
from pprint import pprint

# Keep the backend directory importable when the notebook is opened from here.
backend_dir = os.path.abspath('.')
if backend_dir not in sys.path:
    sys.path.append(backend_dir)

from notebook_setup import setup_django

setup_django()

from common.models import Accounts, Transactions
from core.broker_api_utils import BybitAPI, OKXAPI
from core.crypto_exchange_clients import BybitClient, CryptoExchangeAPIError, OKXClient
from core.crypto_exchange_import import (
    normalize_bybit_spot_execution,
    normalize_okx_spot_fill,
    persist_crypto_exchange_event,
)
from users.models import BybitApiToken, CustomUser, OKXApiToken


In [ ]:
def env_bool(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y'}


def masked(value):
    if not value:
        return '<missing>'
    if len(value) <= 8:
        return '<set>'
    return f'{value[:4]}...{value[-4:]}'


def require_env(*names):
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise RuntimeError(
            'Missing environment variables: ' + ', '.join(missing)
        )


def date_range_ms(days=7):
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)
    return str(int(start.timestamp() * 1000)), str(int(end.timestamp() * 1000))


def optional_int_env(name):
    value = os.getenv(name)
    if value in (None, ''):
        return None
    return int(value)


DEFAULT_USER_ID = int(os.getenv('PM_USER_ID', '1'))
SELECTED_ACCOUNT_ID = optional_int_env('PM_ACCOUNT_ID')
SELECTED_PROVIDER = os.getenv('CRYPTO_API_PROVIDER', 'auto').strip().lower()
LOOKBACK_DAYS = int(os.getenv('CRYPTO_API_LOOKBACK_DAYS', '7'))
ALLOW_DB_WRITES = env_bool('CRYPTO_API_TEST_ALLOW_DB_WRITES', False)

print('Runtime config:')
print('  DEFAULT_USER_ID:', DEFAULT_USER_ID)
print('  SELECTED_PROVIDER:', SELECTED_PROVIDER)
print('  SELECTED_ACCOUNT_ID:', SELECTED_ACCOUNT_ID or '<auto>')
print('  LOOKBACK_DAYS:', LOOKBACK_DAYS)
print('  ALLOW_DB_WRITES:', ALLOW_DB_WRITES)
print('Environment fallback credential preview:')
print('  BYBIT_API_KEY:', masked(os.getenv('BYBIT_API_KEY')))
print('  OKX_API_KEY:', masked(os.getenv('OKX_API_KEY')))


## Database Credential Discovery

This is the preferred path when credentials were saved through the app UI. It finds active Bybit/OKX tokens for `DEFAULT_USER_ID`, finds broker accounts under the token brokers, and chooses a provider/account for adapter tests.

Current local defaults are expected to work with `user_id=1`:

- ByBit broker/account/token under user 1
- OKX broker/account/token under user 1

Set `CRYPTO_API_PROVIDER=bybit` or `CRYPTO_API_PROVIDER=okx` to force one provider. Set `PM_ACCOUNT_ID` only when you want to override the automatically selected account.


In [ ]:
def account_for_broker(broker, preferred_id=None):
    accounts = Accounts.objects.select_related('broker').filter(broker=broker).order_by('id')
    if preferred_id is not None:
        preferred = accounts.filter(id=preferred_id).first()
        if preferred is not None:
            return preferred
    return accounts.first()


def print_token_summary(label, tokens):
    print(f'{label}: {len(tokens)} active token(s)')
    for token in tokens:
        account = account_for_broker(token.broker, SELECTED_ACCOUNT_ID)
        print(
            '  ',
            {
                'token_id': token.id,
                'broker_id': token.broker_id,
                'broker': token.broker.name,
                'api_key': masked(token.api_key),
                'account_id': account.id if account else None,
                'account': account.name if account else None,
                'testnet': getattr(token, 'testnet', None),
                'simulated_trading': getattr(token, 'simulated_trading', None),
            },
        )


if SELECTED_PROVIDER not in {'auto', 'bybit', 'okx'}:
    raise RuntimeError('CRYPTO_API_PROVIDER must be auto, bybit, or okx')

user = CustomUser.objects.get(id=DEFAULT_USER_ID)
bybit_tokens = list(
    BybitApiToken.objects.select_related('broker')
    .filter(user=user, is_active=True)
    .order_by('id')
)
okx_tokens = list(
    OKXApiToken.objects.select_related('broker')
    .filter(user=user, is_active=True)
    .order_by('id')
)

print('Selected user:', {'id': user.id, 'username': user.username, 'email': user.email})
print_token_summary('Bybit', bybit_tokens)
print_token_summary('OKX', okx_tokens)

bybit_db_token = bybit_tokens[0] if bybit_tokens else None
okx_db_token = okx_tokens[0] if okx_tokens else None
bybit_db_account = account_for_broker(bybit_db_token.broker, SELECTED_ACCOUNT_ID) if bybit_db_token else None
okx_db_account = account_for_broker(okx_db_token.broker, SELECTED_ACCOUNT_ID) if okx_db_token else None

if SELECTED_PROVIDER == 'bybit':
    selected_provider = 'bybit'
    selected_token = bybit_db_token
    selected_account = bybit_db_account
elif SELECTED_PROVIDER == 'okx':
    selected_provider = 'okx'
    selected_token = okx_db_token
    selected_account = okx_db_account
elif bybit_db_token:
    selected_provider = 'bybit'
    selected_token = bybit_db_token
    selected_account = bybit_db_account
elif okx_db_token:
    selected_provider = 'okx'
    selected_token = okx_db_token
    selected_account = okx_db_account
else:
    selected_provider = None
    selected_token = None
    selected_account = None

if selected_token is None or selected_account is None:
    raise RuntimeError(
        'No usable stored exchange token/account was found. Save credentials through User Settings and create a broker account first.'
    )

print('Selected adapter target:')
print(
    {
        'provider': selected_provider,
        'token_id': selected_token.id,
        'broker_id': selected_token.broker_id,
        'broker': selected_token.broker.name,
        'account_id': selected_account.id,
        'account': selected_account.name,
        'account_native_id': selected_account.native_id,
    }
)


## Bybit Direct Client Smoke Tests

These cells test signed private requests directly against Bybit. They use the active Bybit token stored in the app database when available, and fall back to `BYBIT_*` environment variables only when no DB token exists.


In [ ]:
if bybit_db_token is not None:
    bybit = BybitClient(
        api_key=bybit_db_token.api_key,
        api_secret=bybit_db_token.get_api_secret(user),
        testnet=bybit_db_token.testnet,
    )
    print(
        'Using DB Bybit token:',
        {
            'token_id': bybit_db_token.id,
            'broker': bybit_db_token.broker.name,
            'account_id': bybit_db_account.id if bybit_db_account else None,
            'testnet': bybit_db_token.testnet,
        },
    )
else:
    require_env('BYBIT_API_KEY', 'BYBIT_API_SECRET')
    bybit = BybitClient(
        api_key=os.environ['BYBIT_API_KEY'],
        api_secret=os.environ['BYBIT_API_SECRET'],
        testnet=env_bool('BYBIT_TESTNET', False),
    )
    print('Using BYBIT_* environment credentials')

bybit_account_type = os.getenv('BYBIT_ACCOUNT_TYPE', 'UNIFIED')
wallet = bybit.get_private(
    '/v5/account/wallet-balance',
    {'accountType': bybit_account_type},
)

print('Bybit wallet response keys:', wallet.keys())
pprint(wallet.get('result', {}))


In [ ]:
start_ms, end_ms = date_range_ms(LOOKBACK_DAYS)
bybit_category = os.getenv('BYBIT_CATEGORY', 'spot')
bybit_execution_params = {
    'category': bybit_category,
    'startTime': start_ms,
    'endTime': end_ms,
}

bybit_executions = []
try:
    bybit_executions = list(bybit.iter_executions(bybit_execution_params))
except CryptoExchangeAPIError as exc:
    print('Bybit execution fetch failed:', exc)

print(f'Fetched {len(bybit_executions)} Bybit execution rows')
if bybit_executions:
    pprint(bybit_executions[0])

In [ ]:
bybit_log_params = {
    'accountType': bybit_account_type,
    'startTime': start_ms,
    'endTime': end_ms,
}

bybit_log_rows = []
try:
    bybit_log_rows = list(bybit.iter_transaction_log(bybit_log_params))
except CryptoExchangeAPIError as exc:
    print('Bybit transaction-log fetch failed:', exc)

print(f'Fetched {len(bybit_log_rows)} Bybit transaction log rows')
if bybit_log_rows:
    pprint(bybit_log_rows[0])

In [ ]:
normalized_bybit_events = []
for payload in bybit_executions[:5]:
    try:
        normalized_bybit_events.append(normalize_bybit_spot_execution(payload))
    except Exception as exc:
        print('Could not normalize Bybit execution:')
        pprint(payload)
        print(exc)

print(f'Normalized {len(normalized_bybit_events)} Bybit events')
if normalized_bybit_events:
    pprint(normalized_bybit_events[0])

## OKX Direct Client Smoke Tests

These cells test signed private requests directly against OKX. They use the active OKX token stored in the app database when available, and fall back to `OKX_*` environment variables only when no DB token exists.


In [ ]:
if okx_db_token is not None:
    okx = OKXClient(
        api_key=okx_db_token.api_key,
        api_secret=okx_db_token.get_api_secret(user),
        passphrase=okx_db_token.get_passphrase(user),
        simulated_trading=okx_db_token.simulated_trading,
    )
    print(
        'Using DB OKX token:',
        {
            'token_id': okx_db_token.id,
            'broker': okx_db_token.broker.name,
            'account_id': okx_db_account.id if okx_db_account else None,
            'simulated_trading': okx_db_token.simulated_trading,
        },
    )
else:
    require_env('OKX_API_KEY', 'OKX_API_SECRET', 'OKX_PASSPHRASE')
    okx = OKXClient(
        api_key=os.environ['OKX_API_KEY'],
        api_secret=os.environ['OKX_API_SECRET'],
        passphrase=os.environ['OKX_PASSPHRASE'],
        simulated_trading=env_bool('OKX_SIMULATED_TRADING', False),
    )
    print('Using OKX_* environment credentials')

balance = okx.get_private('/api/v5/account/balance')
print('OKX balance response keys:', balance.keys())
pprint(balance.get('data', [])[:1])


In [ ]:
start_ms, end_ms = date_range_ms(LOOKBACK_DAYS)
okx_fill_params = {
    'instType': 'SPOT',
    'begin': start_ms,
    'end': end_ms,
}

okx_fills = []
try:
    okx_fills = list(okx.iter_fills_history(okx_fill_params))
except CryptoExchangeAPIError as exc:
    print('OKX fills fetch failed:', exc)

print(f'Fetched {len(okx_fills)} OKX fill rows')
if okx_fills:
    pprint(okx_fills[0])

In [ ]:
normalized_okx_events = []
for payload in okx_fills[:5]:
    try:
        normalized_okx_events.append(normalize_okx_spot_fill(payload))
    except Exception as exc:
        print('Could not normalize OKX fill:')
        pprint(payload)
        print(exc)

print(f'Normalized {len(normalized_okx_events)} OKX events')
if normalized_okx_events:
    pprint(normalized_okx_events[0])

## Test The Project BrokerAPI Adapter Path

This is the preferred integration test when credentials have been stored through User Settings. It uses the selected `Broker` and `Account` instances discovered from the database above and exercises the same adapter path used by Direct Import.

The cell fetches normalized events only. It does not persist transactions.


In [ ]:
if selected_provider == 'bybit':
    adapter = BybitAPI()
elif selected_provider == 'okx':
    adapter = OKXAPI()
else:
    raise RuntimeError('No selected provider. Run Database Credential Discovery first.')

account = selected_account
if account is None:
    raise RuntimeError('No selected account. Run Database Credential Discovery first.')

date_to = datetime.now(timezone.utc).date().isoformat()
date_from = (datetime.now(timezone.utc).date() - timedelta(days=LOOKBACK_DAYS)).isoformat()

print(
    'Fetching via BrokerAPI:',
    {
        'adapter': adapter.__class__.__name__,
        'broker': account.broker.name,
        'account_id': account.id,
        'account': account.name,
        'date_from': date_from,
        'date_to': date_to,
    },
)

await adapter.connect(user)
adapter_events = []
async for event in adapter.get_transactions(account, date_from=date_from, date_to=date_to):
    adapter_events.append(event)
await adapter.disconnect()

print(f'Fetched {len(adapter_events)} normalized events through {adapter.__class__.__name__}')
if adapter_events:
    pprint(adapter_events[0])


## Optional: Persist One Normalized Event

This writes `Transactions` rows. Keep `CRYPTO_API_TEST_ALLOW_DB_WRITES=0` unless you are intentionally testing persistence in a disposable database or with a transaction you are prepared to delete.

The importer is idempotent by provider/account/event id, but this is still a database write.

In [ ]:
if not ALLOW_DB_WRITES:
    raise RuntimeError(
        'Database writes are disabled. Set CRYPTO_API_TEST_ALLOW_DB_WRITES=1 to run this cell.'
    )

if adapter_events:
    events_to_persist = adapter_events
    account_to_persist = selected_account
elif normalized_bybit_events and bybit_db_account:
    events_to_persist = normalized_bybit_events
    account_to_persist = bybit_db_account
elif normalized_okx_events and okx_db_account:
    events_to_persist = normalized_okx_events
    account_to_persist = okx_db_account
else:
    raise RuntimeError('No normalized events with a matching DB account are available to persist')

created = persist_crypto_exchange_event(events_to_persist[0], user, account_to_persist)
print(f'Created {len(created)} Transactions rows')
for tx in created:
    print(tx.id, tx.type, tx.security, tx.quantity, tx.price, tx.import_provider, tx.import_event_id)


## Troubleshooting Notes

- `No usable stored exchange token/account was found`: confirm migrations are applied, credentials are saved through User Settings, and a broker account exists under the same broker.
- `no such column: common_transactions.import_provider`: run `python manage.py migrate` from the backend directory.
- Bybit `permission denied` or similar: check that the key has read access for account, wallet, transaction log, and execution history.
- Bybit testnet failures with mainnet keys: testnet and mainnet keys are not interchangeable.
- OKX `Invalid Sign`: verify the API secret and passphrase exactly. Passphrases are user-defined and case-sensitive.
- OKX simulated failures: use `simulated_trading=True` only with demo/simulated credentials.
- Empty results are valid if there were no fills/log entries in the selected date range. Increase `CRYPTO_API_LOOKBACK_DAYS`.
- Keep raw payloads when normalization fails; those payloads are the best fixtures for extending the normalizers.
